# Preparing variable-length jet constituents

This lesson turns the list-valued Parquet columns into a scalable representation shared by
three advanced classifiers. It uses every constituent, prevents event leakage, and records
the exact source-data fingerprint. Rerunning the generator with ten times more events is
detected automatically.


## 1. Environment


In [ ]:
import importlib.util
required = ['numpy', 'pandas', 'pyarrow', 'matplotlib', 'sklearn', 'torch']
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(
        f"Missing packages: {missing}. From a terminal in this directory run "
        "./setup_student_env.sh, restart Jupyter with `henv . -x jupyter lab`, "
        "and select the Quark/Gluon Constituent ML kernel."
    )

import json, os, time
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import torch
import qg_constituent_ml as qg

DEVICE = qg.choose_device()
RUN_MODE = os.getenv('QG_RUN_MODE', 'quick')
SOURCE = Path(os.getenv('QG_INPUT_PATH', 'data/inclusive_jets.parquet'))
print(f'PyTorch {torch.__version__}; built with CUDA {torch.version.cuda}')
print(f'device={DEVICE}' + (f'; GPU={torch.cuda.get_device_name(0)}' if DEVICE.type == 'cuda' else ''))
print(f'run mode={RUN_MODE}; source={SOURCE}')


## 2. Representation

For particle $i$, use $z_i=p_{T,i}/\sum_jp_{T,j}$ and coordinates relative to the jet.
Continuous inputs are `log(z)`, $\Delta\eta$, wrapped $\Delta\phi$, and `log(ΔR)`.
Particle identity is categorical—not an ordinal PDG number. Stable event hashes create
70/15/15 train/validation/test partitions, and normalization sees train particles only.


In [ ]:
prepared = qg.prepare_dataset(SOURCE)
manifest = qg.load_manifest(prepared)
arrays = qg.load_arrays(prepared)
print(f'Prepared directory: {prepared}')
print(json.dumps(manifest, indent=2))
assert arrays['offsets'][-1] == manifest['n_constituents']
assert len(arrays['labels']) == manifest['n_jets']


## 3. Inspect statistics and particle categories


In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4))
axes[0].hist(arrays['n_constituents'],bins=range(0,int(arrays['n_constituents'].max())+2),histtype='step')
axes[0].set(xlabel='constituents per jet',ylabel='jets',title='Dynamic batch lengths')
counts=np.asarray(manifest['normalization']['mean']); axes[1].bar(qg.CONTINUOUS_FEATURES,counts)
axes[1].tick_params(axis='x',rotation=25); axes[1].set(title='Training-set raw means')
plt.tight_layout(); plt.show()


## 4. Validate event isolation and dynamic padding


In [ ]:
events=np.asarray(arrays['event_ids']); splits=np.asarray(arrays['splits'])
sets=[set(events[splits==i]) for i in range(3)]
assert sets[0].isdisjoint(sets[1]) and sets[0].isdisjoint(sets[2]) and sets[1].isdisjoint(sets[2])
JetDataset=qg.make_dataset_classes(); ds=JetDataset(prepared,split='train')
batch=qg.collate_jets([ds[i] for i in range(min(8,len(ds)))])
assert batch['mask'].sum().item() == sum(len(ds[i]['features']) for i in range(min(8,len(ds))))
print({k:tuple(v.shape) for k,v in batch.items() if hasattr(v,'shape')})


## 5. Scaling the statistics

Change the visible generator setting or launch it with `QG_N_EVENTS=200000` for ten times
the default event count. The resulting Parquet SHA-256 changes; this setup creates a new
memory-mappable prepared directory, while model checkpoints trained on older statistics
remain separately identified.
